# Module 1: Dataset Acquisition & Initial Setup

## 1.1 · Project environment setup: configuration

In [ ]:
RUN = {"unsw_nb15": True, "cic_ids2017": True, "bot_iot": True}

DATASETS = {
    "bot_iot":     {"handle": "shamstahzib/bot-iot-countershield-extract", "version": None, "name": "BoT-IoT", "prebuilt": "bot_iot.parquet"},
    "cic_ids2017": {"handle": "shamstahzib/cicids2017-dataset", "version": None, "name": "CIC-IDS2017"},
    "unsw_nb15":   {"handle": "shamstahzib/unsw-nb15-train-test", "version": None, "name": "UNSW-NB15"},
}

USE_DRIVE     = True
PROJECT_DIR   = "CounterShield"
FORCE_REBUILD = False
HASH_FILES    = True

BOTIOT_MAX_PER_CLASS = 300_000
BOTIOT_FILE_PATTERN  = None
BOTIOT_CHUNKSIZE     = 1_000_000
CIC_DROP_DUPLICATES  = False
SEED = 42

## 1.1 · Project environment setup: install & helpers

In [ ]:
!pip -q install -U kagglehub pyarrow
import kagglehub, sys, datetime as dt

import os, re, gc, json, time, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd

def list_data_files(root, exts=(".csv", ".parquet")):
    root = Path(root)
    return sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts)

def human(n):
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024: return f"{n:,.1f} {u}"
        n /= 1024
    return f"{n:,.1f} PB"

def sha256_file(path, block=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

def clean_columns(df):
    df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in df.columns]
    return df

def find_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

def norm_label(x):
    s = str(x).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    s = re.sub(r"_+", "_", s)
    if s in {"benign", "normal", "0", "", "nan", "none"}:
        return "benign"
    return s

def label_family(c):
    if c == "benign": return "benign"
    if "ddos" in c: return "ddos"
    if "dos" in c: return "dos"
    if any(k in c for k in ("portscan", "reconnaissance", "recon", "scan", "fingerprint", "analysis")):
        return "recon"
    if any(k in c for k in ("patator", "brute", "password")) and "web" not in c:
        return "brute_force"
    if any(k in c for k in ("web_attack", "xss", "sql")): return "web_attack"
    if c in ("bot", "botnet"): return "botnet"
    if any(k in c for k in ("theft", "exfiltration", "keylog")): return "exfiltration_theft"
    if any(k in c for k in ("backdoor", "worm")): return "malware_backdoor"
    if any(k in c for k in ("exploit", "shellcode", "infiltration", "heartbleed")): return "exploit"
    return "other"

def downcast(df):
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = df[c].astype("float32")
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

def replace_inf(df):
    num = df.select_dtypes(include=[np.number]).columns
    df[num] = df[num].replace([np.inf, -np.inf], np.nan)
    return df

def add_labels(df, cls_series, binary_series, source, split="all"):
    df["y_class"] = cls_series.map(norm_label).astype("category")
    df["y_binary"] = binary_series.astype("int8")
    df["y_family"] = df["y_class"].astype(str).map(label_family).astype("category")
    df["source_dataset"] = source
    df["split"] = split
    return df

def read_any(path, **kw):
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False, **kw)

def process_unsw(root):
    files = list_data_files(root)
    tt = [f for f in files if re.search(r"train|test", f.name, re.I)]
    files = tt or files
    print("UNSW-NB15 files used:", [f.name for f in files])
    parts = []
    for f in files:
        df = clean_columns(read_any(f))
        lab = find_col(df, ["label"])
        cat = find_col(df, ["attack_cat"])
        if lab is None and cat is None:
            print(f"  ! skipping {f.name}: no label/attack_cat column"); continue
        split = "train" if re.search(r"train", f.name, re.I) else ("test" if re.search(r"test", f.name, re.I) else "all")
        cls = df[cat].astype(str) if cat else df[lab].map({0: "benign", 1: "attack"})
        if lab is not None:
            binary = pd.to_numeric(df[lab], errors="coerce").fillna(1)
        else:
            binary = (cls.map(norm_label) != "benign")
        parts.append(add_labels(df, cls, binary, "unsw_nb15", split))
    out = pd.concat(parts, ignore_index=True)
    return downcast(replace_inf(out))

def process_cic(root, drop_duplicates=False):
    files = list_data_files(root)
    print(f"CIC-IDS2017: {len(files)} files")
    parts = []
    for f in files:
        df = clean_columns(read_any(f))
        lab = find_col(df, ["Label", "label", "Attack", "attack_cat"])
        if lab is None:
            print(f"  ! skipping {f.name}: no label column"); continue
        cls = df[lab].astype(str)
        binary = cls.map(norm_label) != "benign"
        df = add_labels(df, cls, binary, "cic_ids2017", "all")
        df["source_file"] = f.stem
        parts.append(downcast(replace_inf(df)))
        print(f"  {f.name:<55} {len(df):>9,} rows")
    out = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    n0 = len(out)
    num = out.select_dtypes(include=[np.number]).columns
    out = out.dropna(subset=num).reset_index(drop=True)
    if drop_duplicates:
        out = out.drop_duplicates().reset_index(drop=True)
    print(f"  dropped {n0 - len(out):,} rows with NaN/inf" + (" or duplicates" if drop_duplicates else ""))
    for c in ("y_class", "y_family", "source_file"):
        out[c] = out[c].astype("category")
    return out

BOTIOT_FULL_COLUMNS = [
    "pkSeqID", "stime", "flgs", "proto", "saddr", "sport", "daddr", "dport", "pkts", "bytes",
    "state", "ltime", "seq", "dur", "mean", "stddev", "smac", "dmac", "sum", "min", "max",
    "soui", "doui", "sco", "dco", "spkts", "dpkts", "sbytes", "dbytes", "rate", "srate",
    "drate", "attack", "category", "subcategory",
]

def select_botiot_files(root, pattern=None):
    files = [f for f in list_data_files(root, (".csv",))]
    feat = [f for f in files if re.search(r"feature_names|features", f.name, re.I)]
    data = [f for f in files if f not in feat]
    if pattern:
        chosen = sorted(Path(root).rglob(pattern))
    else:
        full = [f for f in data if re.search(r"(^data_\d+|Dataset_\d+)\.csv$", f.name, re.I)]
        pc5 = [f for f in data if re.search(r"5pc_\d+\.csv$", f.name, re.I)]
        chosen = full or pc5 or data
    key = lambda p: [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.name)]
    return sorted(chosen, key=key), (feat[0] if feat else None)

def _botiot_names(path, feature_file):
    first = pd.read_csv(path, nrows=1, header=None, low_memory=False)
    row = [str(v).strip().strip('"') for v in first.iloc[0].tolist()]
    if any(v.lower() in {"pkseqid", "attack", "category", "proto"} for v in row):
        return 0, None
    if feature_file is not None:
        names = [c.strip() for c in pd.read_csv(feature_file, header=None).iloc[0].astype(str).tolist() if str(c).strip()]
        if len(names) == len(row):
            return None, names
    if len(row) == len(BOTIOT_FULL_COLUMNS):
        return None, BOTIOT_FULL_COLUMNS
    raise ValueError(f"{path.name}: no header and {len(row)} columns - set BOTIOT_FILE_PATTERN or column names manually")

def _to_port(s):
    s = s.astype(str).str.strip()
    hexm = s.str.startswith("0x")
    out = pd.to_numeric(s.where(~hexm), errors="coerce")
    if hexm.any():
        out[hexm] = s[hexm].map(lambda v: int(v, 16) if re.fullmatch(r"0x[0-9a-fA-F]+", v) else np.nan)
    return out.astype("float32")

def process_botiot(root, max_per_class=300_000, chunksize=1_000_000, pattern=None, seed=42):
    files, feature_file = select_botiot_files(root, pattern)
    if not files:
        raise FileNotFoundError("No BoT-IoT CSV files found")
    print(f"BoT-IoT: {len(files)} CSV files, total {human(sum(f.stat().st_size for f in files))}")
    specs = {f: _botiot_names(f, feature_file) for f in files}

    def iter_chunks(usecols=None):
        for f in files:
            header, names = specs[f]
            kw = dict(header=header, names=names, chunksize=chunksize, low_memory=False)
            if usecols is not None:
                kw["usecols"] = lambda c: c.strip() in usecols
            for ch in pd.read_csv(f, **kw):
                yield f, clean_columns(ch)

    counts = pd.Series(dtype="int64")
    for _, ch in iter_chunks(usecols={"category"}):
        counts = counts.add(ch["category"].astype(str).str.strip().value_counts(), fill_value=0)
    counts = counts.astype("int64").sort_values(ascending=False)
    print("  full class counts:\n" + counts.to_string())
    frac = (max_per_class / counts).clip(upper=1.0) if max_per_class else counts * 0 + 1.0

    rng = np.random.default_rng(seed)
    parts = []
    for f, ch in iter_chunks():
        cat = ch["category"].astype(str).str.strip()
        keep = rng.random(len(ch)) < cat.map(frac).fillna(1.0).to_numpy()
        if keep.any():
            parts.append(ch.loc[keep])
    df = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    for c in ("sport", "dport"):
        if c in df.columns: df[c] = _to_port(df[c])
    lab = find_col(df, ["attack"])
    cls = df["category"].astype(str).str.strip()
    binary = pd.to_numeric(df[lab], errors="coerce").fillna(1) if lab else (cls.map(norm_label) != "benign")
    df = add_labels(df, cls, binary, "bot_iot", "all")
    if "subcategory" in df.columns:
        df["subcategory"] = df["subcategory"].astype(str).str.strip().astype("category")
    df = downcast(replace_inf(df))
    print(f"  sampled {len(df):,} rows (max_per_class={max_per_class})")
    return df, counts

ID_OR_LEAKY_COLS = {
    "unsw_nb15": ["id", "label", "attack_cat"],
    "cic_ids2017": ["Flow ID", "Source IP", "Src IP", "Destination IP", "Dst IP",
                    "Timestamp", "Label", "Attack", "source_file"],
    "bot_iot": ["pkSeqID", "stime", "ltime", "seq", "saddr", "daddr", "smac", "dmac",
                "soui", "doui", "sco", "dco", "attack", "category", "subcategory"],
}
LABEL_COLS = ["y_class", "y_binary", "y_family", "source_dataset", "split"]

def get_xy(df, target="y_binary"):
    src = str(df["source_dataset"].iloc[0])
    drop = [c for c in ID_OR_LEAKY_COLS.get(src, []) + LABEL_COLS if c in df.columns]
    X = df.drop(columns=drop)
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]) or pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype("category").cat.codes.astype("int32")
    X = X.fillna(0)
    return X, df[target]

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


## 1.1 · Project environment setup: Google Drive

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_DIR
else:
    BASE = Path("/content") / PROJECT_DIR
PROCESSED = BASE / "processed"
PROV_DIR  = BASE / "processed" / "provenance"
for p in (PROCESSED, PROV_DIR): p.mkdir(parents=True, exist_ok=True)
print("Outputs ->", BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Outputs -> /content/drive/MyDrive/CounterShield


## 1.2 · Dataset acquisition: Kaggle authentication

In [ ]:
def setup_kaggle_auth():
    keys = ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN")
    try:
        from google.colab import userdata
        for k in keys:
            try:
                v = userdata.get(k)
                if v: os.environ[k] = v.strip()
            except Exception:
                pass
    except ImportError:
        pass
    ok = lambda: os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    if ok(): return "Colab secrets"
    for p in (BASE / "kaggle.json", Path("~/.kaggle/kaggle.json").expanduser()):
        if p.exists():
            creds = json.loads(p.read_text()); break
    else:
        from google.colab import files
        print("Upload your kaggle.json")
        up = files.upload()
        creds = json.loads(next(iter(up.values())).decode())
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]
    kdir = Path("~/.kaggle").expanduser(); kdir.mkdir(exist_ok=True)
    (kdir / "kaggle.json").write_text(json.dumps(creds)); os.chmod(kdir / "kaggle.json", 0o600)
    return "kaggle.json"

print("Kaggle auth via:", setup_kaggle_auth())

Kaggle auth via: Colab secrets


## 1.2 · Dataset acquisition: download

In [ ]:
def out_path(key): return PROCESSED / f"{key}.parquet"

RAW, PROVENANCE, PREBUILT = {}, {}, {}
prov_file = PROV_DIR / "provenance.json"
if prov_file.exists():
    PROVENANCE = json.loads(prov_file.read_text())

for key, meta in DATASETS.items():
    if not RUN.get(key): continue
    force = FORCE_REBUILD if isinstance(FORCE_REBUILD, bool) else key in FORCE_REBUILD
    if out_path(key).exists() and not force:
        print(f"{meta['name']}: processed file already on Drive -> skip download"); continue
    handle = meta["handle"] + (f"/versions/{meta['version']}" if meta["version"] else "")
    t0 = time.time()
    path = Path(kagglehub.dataset_download(handle))
    if meta.get("prebuilt"):
        import shutil; shutil.copy(path / meta["prebuilt"], out_path(key))
        src_prov = path / f"provenance_{key}.json"
        PROVENANCE[key] = json.loads(src_prov.read_text()) if src_prov.exists() else {"name": meta["name"]}
        PROVENANCE[key]["published_extract"] = f"https://www.kaggle.com/datasets/{meta['handle']}"
        PREBUILT[key] = pd.read_parquet(out_path(key), columns=["split", "y_family", "y_class"]).groupby(
            ["split", "y_family", "y_class"], observed=True).size().rename("rows").reset_index()
        print(f"{meta['name']}: copied prebuilt {meta['prebuilt']} from {meta['handle']}"); continue
    RAW[key] = path
    m = re.search(r"versions/(\d+)", str(path))
    files = list_data_files(path)
    print(f"{meta['name']}: {len(files)} data files, {human(sum(f.stat().st_size for f in files))}  ({time.time()-t0:.0f}s)")
    PROVENANCE[key] = {
        "name": meta["name"],
        "kaggle_handle": meta["handle"],
        "kaggle_url": f"https://www.kaggle.com/datasets/{meta['handle']}",
        "kaggle_version": int(m.group(1)) if m else None,
        "downloaded_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "kagglehub_version": kagglehub.__version__,
        "files": [{"file": str(f.relative_to(path)), "bytes": f.stat().st_size} for f in files],
    }

BoT-IoT: processed file already on Drive -> skip download


100%|██████████| 230M/230M [00:04<00:00, 53.4MB/s]

Extracting files...


CIC-IDS2017: 8 data files, 843.7 MB  (22s)


100%|██████████| 11.9M/11.9M [00:00<00:00, 48.6MB/s]

Extracting files...


UNSW-NB15: 2 data files, 45.5 MB  (2s)


## 1.3 · Dataset inventory

In [ ]:
for key, path in RAW.items():
    print(f"\n=== {DATASETS[key]['name']}  ({path})")
    for f in list_data_files(path)[:40]:
        print(f"  {human(f.stat().st_size):>10}  {f.relative_to(path)}")
    first = list_data_files(path)[0]
    peek = pd.read_parquet(first).head(3) if first.suffix == ".parquet" else pd.read_csv(first, nrows=3, low_memory=False)
    display(peek)


=== CIC-IDS2017  (/root/.cache/kagglehub/datasets/shamstahzib/cicids2017-dataset/versions/2)
     73.6 MB  CICIDS2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
     73.3 MB  CICIDS2017/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
     55.6 MB  CICIDS2017/Friday-WorkingHours-Morning.pcap_ISCX.csv
    168.7 MB  CICIDS2017/Monday-WorkingHours.pcap_ISCX.csv
     79.3 MB  CICIDS2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
     49.6 MB  CICIDS2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
    128.8 MB  CICIDS2017/Tuesday-WorkingHours.pcap_ISCX.csv
    214.7 MB  CICIDS2017/Wednesday-workingHours.pcap_ISCX.csv


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN



=== UNSW-NB15  (/root/.cache/kagglehub/datasets/shamstahzib/unsw-nb15-train-test/versions/2)
     14.7 MB  UNSW_NB15_testing-set.csv
     30.8 MB  UNSW_NB15_training-set.csv


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0


## 1.4 · Initial dataset verification & harmonised Parquet

In [ ]:
SUMMARY = dict(PREBUILT)

def save(key, df):
    df.to_parquet(out_path(key), index=False)
    SUMMARY[key] = df.groupby(["split", "y_family", "y_class"], observed=True).size().rename("rows").reset_index()
    print(f"saved {out_path(key)}  {df.shape[0]:,} rows x {df.shape[1]} cols  ({human(out_path(key).stat().st_size)})")

if "unsw_nb15" in RAW:
    unsw = process_unsw(RAW["unsw_nb15"]); save("unsw_nb15", unsw)
    display(unsw.groupby(["split", "y_class"], observed=True).size().unstack(0))
    del unsw; gc.collect()

UNSW-NB15 files used: ['UNSW_NB15_testing-set.csv', 'UNSW_NB15_training-set.csv']
saved /content/drive/MyDrive/CounterShield/processed/unsw_nb15.parquet  257,673 rows x 50 cols  (15.6 MB)


split,test,train
y_class,,
analysis,677,2000
backdoor,583,1746
benign,37000,56000
dos,4089,12264
exploits,11132,33393
fuzzers,6062,18184
generic,18871,40000
reconnaissance,3496,10491
shellcode,378,1133


In [ ]:
if "cic_ids2017" in RAW:
    cic = process_cic(RAW["cic_ids2017"], drop_duplicates=CIC_DROP_DUPLICATES); save("cic_ids2017", cic)
    PROVENANCE["cic_ids2017"]["processing"] = {"dropped_nan_inf": True, "drop_duplicates": CIC_DROP_DUPLICATES}
    display(cic["y_class"].value_counts().to_frame("rows"))
    del cic; gc.collect()

CIC-IDS2017: 8 files
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv          225,745 rows
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv      286,467 rows
  Friday-WorkingHours-Morning.pcap_ISCX.csv                 191,033 rows
  Monday-WorkingHours.pcap_ISCX.csv                         529,918 rows
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv   288,602 rows
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv    170,366 rows
  Tuesday-WorkingHours.pcap_ISCX.csv                        445,909 rows
  Wednesday-workingHours.pcap_ISCX.csv                      692,703 rows
  dropped 2,867 rows with NaN/inf
saved /content/drive/MyDrive/CounterShield/processed/cic_ids2017.parquet  2,827,876 rows x 85 cols  (258.2 MB)


,rows
y_class,
benign,2271320
dos_hulk,230124
portscan,158804
ddos,128025
dos_goldeneye,10293
ftp_patator,7935
ssh_patator,5897
dos_slowloris,5796
dos_slowhttptest,5499


In [ ]:
if "bot_iot" in RAW:
    files, _ = select_botiot_files(RAW["bot_iot"], BOTIOT_FILE_PATTERN)
    print("Using:", [f.name for f in files][:5], "..." if len(files) > 5 else "")
    bot, full_counts = process_botiot(RAW["bot_iot"], BOTIOT_MAX_PER_CLASS, BOTIOT_CHUNKSIZE, BOTIOT_FILE_PATTERN, SEED)
    save("bot_iot", bot)
    PROVENANCE["bot_iot"]["processing"] = {
        "files_used": [f.name for f in files],
        "full_class_counts": {k: int(v) for k, v in full_counts.items()},
        "sampling": f"stratified by category, max {BOTIOT_MAX_PER_CLASS} rows/class, seed {SEED}",
    }
    display(bot.groupby(["y_class", "subcategory"], observed=True).size().to_frame("rows"))
    del bot; gc.collect()

## 1.4 · Initial dataset verification: provenance record

In [ ]:
if HASH_FILES:
    for key, path in RAW.items():
        print("hashing", DATASETS[key]["name"], "...")
        for rec in PROVENANCE[key]["files"]:
            rec["sha256"] = sha256_file(path / rec["file"])

for key in RAW:
    PROVENANCE[key]["processed_file"] = str(out_path(key).relative_to(BASE))
    PROVENANCE[key]["processed_sha256"] = sha256_file(out_path(key))
PROVENANCE["_environment"] = {"python": sys.version.split()[0], "pandas": pd.__version__,
                              "numpy": np.__version__, "platform": platform.platform(), "seed": SEED}
prov_file.write_text(json.dumps(PROVENANCE, indent=2))

if SUMMARY:
    summary = pd.concat([s.assign(dataset=k) for k, s in SUMMARY.items()], ignore_index=True)
    old_cd = PROV_DIR / "class_distribution.csv"
    if old_cd.exists():
        old = pd.read_csv(old_cd); summary = pd.concat([old[~old.dataset.isin(list(SUMMARY))], summary], ignore_index=True)
    summary.to_csv(PROV_DIR / "class_distribution.csv", index=False)
    display(summary.pivot_table(index="y_family", columns="dataset", values="rows", aggfunc="sum", fill_value=0))
print("provenance ->", prov_file)
print({k: v.get("kaggle_version") for k, v in PROVENANCE.items() if not k.startswith("_")})

hashing CIC-IDS2017 ...
hashing UNSW-NB15 ...


dataset,bot_iot,cic_ids2017,unsw_nb15
y_family,,,
benign,9543,2271320,93000
botnet,0,1956,0
brute_force,0,13832,0
ddos,301150,128025,0
dos,299665,251712,16353
exfiltration_theft,1587,0,0
exploit,0,47,46036
malware_backdoor,0,0,2503
other,0,0,83117


provenance -> /content/drive/MyDrive/CounterShield/processed/provenance/provenance.json
{'bot_iot': 1, 'cic_ids2017': 2, 'unsw_nb15': 2}
